In [ ]:
import zipfile

import numpy as np
import pandas as pd


## 1. Load the profile table

Output of `01-2_work_experience.ipynb`.


In [ ]:
df = pd.read_csv("/path/to/CausalFair/Resume/job-distribution/processed_job_data_0102_with_exp_pred.csv")
df["age_bin"].value_counts()

## 2. First-name pool from the SSA baby-name archive

Needs `sources/names.zip` — see `sources/README.md`.


In [ ]:
CURRENT_YEAR = 2023
NAMES_ZIP_PATH = "./sources/names.zip"
TOP_N = None  # 1000

name_pool_by_year = {}

def load_ssa_year_from_zip(year: int) -> pd.DataFrame:
    with zipfile.ZipFile(NAMES_ZIP_PATH) as z:
        fname = f"yob{year}.txt"
        with z.open(fname) as f:
            df = pd.read_csv(f, names=["name", "sex", "count"], dtype={"name": str, "sex": str, "count": int})
    df["year"] = year
    df["age"] = CURRENT_YEAR - year
    return df

def topN_with_p(df_year: pd.DataFrame) -> pd.DataFrame:
    totals = df_year.groupby("sex")["count"].transform("sum")
    df = df_year.copy()
    df["p_name_given_sex_year"] = df["count"] / totals
    df = df.sort_values(["sex", "count", "name"], ascending=[True, False, True])
    df["rank_in_year"] = df.groupby("sex").cumcount() + 1
    if TOP_N is not None:
        return df[df["rank_in_year"] <= TOP_N].reset_index(drop=True)
    return df.reset_index(drop=True)

def get_year_pool(year: int):
    if year not in name_pool_by_year:
        df_raw = load_ssa_year_from_zip(year)
        name_pool_by_year[year] = topN_with_p(df_raw)
    return name_pool_by_year[year]

for year in range(CURRENT_YEAR - 45, CURRENT_YEAR - 16):
    get_year_pool(year)

df_name_pool = pd.concat(name_pool_by_year.values(), axis=0).reset_index(drop=True)

age_bins = [17, 25, 35, 45]
age_labels = ["[17, 25)", "[25, 35)", "[35, 45)"]
df_name_pool["age_group"] = pd.cut(df_name_pool["age"], bins=age_bins, right=False, labels=age_labels)
df_name_pool = df_name_pool[df_name_pool["age"] != 45].reset_index(drop=True)

df_name_pool_using_age_group = (
    df_name_pool
    .groupby(["age_group", "sex", "name"], observed=True)
    .agg(count=("count", "sum"))
    .reset_index()
)

df_name_pool_using_age_group["p_name_given_age_group_sex"] = (
    df_name_pool_using_age_group["count"]
    / df_name_pool_using_age_group.groupby(["age_group", "sex"], observed=True)["count"].transform("sum")
)

df_name_pool_using_age_group["p_age_group_sex_given_name"] = (
    df_name_pool_using_age_group["count"]
    / df_name_pool_using_age_group.groupby("name")["count"].transform("sum")
)

# -------------------------------------------------
# P(age_group | name)
# -------------------------------------------------
df_name_pool_using_age_group["p_age_group_given_name"] = (
    df_name_pool_using_age_group
    .groupby(["name", "age_group"], observed=True)["count"]
    .transform("sum")
    / df_name_pool_using_age_group.groupby("name")["count"].transform("sum")
)

# -------------------------------------------------
# P(sex | name)
# -------------------------------------------------
df_name_pool_using_age_group["p_sex_given_name"] = (
    df_name_pool_using_age_group
    .groupby(["name", "sex"], observed=True)["count"]
    .transform("sum")
    / df_name_pool_using_age_group.groupby("name")["count"].transform("sum")
)


df_name_pool_using_age_group["first_name_group"] = (
    df_name_pool_using_age_group
    .groupby(["age_group", "sex"], observed=True)["p_age_group_sex_given_name"]
    .transform(lambda x: pd.qcut(x, q=3, labels=["rare", "medium", "common"], duplicates="drop"))
)

df_name_pool_using_age_group[df_name_pool_using_age_group["name"] == "John"]

## 3. First-name grouping labels


In [ ]:
# --- First-name grouping labels -------------------------------------------------
# Paper, Appendix "Grouping of High Cardinality Attributes":
#   gender-typicality : arg-max sex when P(sex | name) >= 0.75, else "neutral"
#   age-typicality    : arg-max age group when P(age_group | name) >= 0.5, else "neutral"
SEX_THRESHOLD = 0.75
AGE_THRESHOLD = 0.5


def assign_age_label(group):
    if (group["p_age_group_given_name"] >= AGE_THRESHOLD).any():
        return group.loc[group["p_age_group_given_name"].idxmax(), "age_group"]
    return "neutral"


def assign_sex_label(group):
    if (group["p_sex_given_name"] >= SEX_THRESHOLD).any():
        return group.loc[group["p_sex_given_name"].idxmax(), "sex"]
    return "neutral"


name_age_label = df_name_pool_using_age_group.groupby("name", observed=True).apply(assign_age_label)
name_sex_label = df_name_pool_using_age_group.groupby("name", observed=True).apply(assign_sex_label)

df_name_pool_using_age_group["first_name_age_group"] = df_name_pool_using_age_group["name"].map(name_age_label)
df_name_pool_using_age_group["first_name_sex"] = df_name_pool_using_age_group["name"].map(name_sex_label)

df_name_pool_using_age_group.to_csv("./name_grouping_first_final.csv", index=False)
print(df_name_pool_using_age_group["first_name_sex"].value_counts().to_dict())
print(df_name_pool_using_age_group["first_name_age_group"].value_counts().to_dict())


## 4. Draw a first name for each individual

Sampled from the SSA pool for that person's exact birth year and sex.


In [ ]:
def sample_first_name(row, mode="exact_age"):
    age = int(row["age"])
    sex = row["sex"]
    sex = "M" if sex == 1 else "F" 
    
    birth_year = CURRENT_YEAR - age
    df_year = get_year_pool(birth_year)
    pool = df_year[df_year.sex == sex]
    # if sum of p is not 1, raise error
    if not np.isclose(pool["p_name_given_sex_year"].sum(), 1.0):
        raise ValueError(f"Probability sum for year {birth_year} and sex {sex} is not 1.")

    return np.random.choice(pool["name"].values, p=pool["p_name_given_sex_year"].values)

df["first_name"] = df.apply(
    lambda r: sample_first_name(r, mode="exact_age"),
    axis=1
)

## 5. Surname pool from the Census surname table

Needs `sources/Names_2010Census_Top1000.xlsx`.


In [ ]:
RACE_TO_CENSUS_COL = {
    "White": "PERCENT NON-HISPANIC OR LATINO WHITE ALONE",
    "Black": "PERCENT NON-HISPANIC OR LATINO BLACK OR AFRICAN AMERICAN ALONE",
    "Asian/Pacific Islander": (
        "PERCENT NON-HISPANIC OR LATINO ASIAN AND NATIVE HAWAIIAN "
        "AND OTHER PACIFIC ISLANDER ALONE"
    ),
    # "AIAN": "PERCENT NON-HISPANIC OR LATINO AMERICAN INDIAN AND ALASKA NATIVE ALONE",
    # "Two races": "PERCENT NON-HISPANIC OR LATINO TWO OR MORE RACES",
}

surname_df = pd.read_excel("./sources/Names_2010Census_Top1000.xlsx")
surname_df["surname"] = surname_df["SURNAME"].str.title()

surname_pools = {}
for race, col in RACE_TO_CENSUS_COL.items():
    print(race, col)
    df_surname = surname_df[["surname", col]].copy()

    # string → float (safe)
    df_surname[col] = pd.to_numeric(df_surname[col], errors="coerce")

    df_surname[col] = df_surname[col] / 100.0
    df_surname = df_surname[df_surname[col] > 0]
    df_surname["p"] = df_surname[col] / df_surname[col].sum()
    surname_pools[race] = df_surname[["surname", "p"]]

In [ ]:
# Long-format surname pool: one row per (surname, race)
df_surname_pool = pd.concat(
    [pool.assign(race=race) for race, pool in surname_pools.items()], ignore_index=True
)

# P(surname | race) is already in column "p"
df_surname_pool.rename(columns={"p": "p_surname_given_race"}, inplace=True)

# P(race | surname)
df_surname_pool["p_race_given_surname"] = (
    df_surname_pool["p_surname_given_race"]
    / df_surname_pool.groupby("surname")["p_surname_given_race"].transform("sum")
)


## 6. Surname grouping labels


In [ ]:
# --- Surname grouping labels ----------------------------------------------------
# Paper, Appendix: assign a surname to Asian / Black / White when
# P(race | surname) >= 0.5; everything else becomes "neutral".
RACE_THRESHOLD = 0.5


def assign_race_label(group):
    if (group["p_race_given_surname"] >= RACE_THRESHOLD).any():
        return group.loc[group["p_race_given_surname"].idxmax(), "race"]
    return "neutral"


surname_label = df_surname_pool.groupby("surname", observed=True).apply(assign_race_label)
df_surname_pool["surname_race_label"] = df_surname_pool["surname"].map(surname_label)

df_surname_pool.to_csv("./name_grouping_surname_final.csv", index=False)
print(df_surname_pool["surname_race_label"].value_counts().to_dict())


## 7. Draw a surname for each individual


In [ ]:
def sample_surname(race_name):
    race = race_name

    if race not in surname_pools:
        # raise exception
        raise ValueError(f"Race '{race}' not found in surname pools.")
    else:
        pool = surname_pools[race]

    idx = np.random.choice(len(pool), p=pool["p"].values)
    surname = pool.iloc[idx]["surname"]
    p = pool.iloc[idx]["p"]

    return surname
df[["last_name"]] = df["race"].apply(
    lambda r: pd.Series(sample_surname(r))
)
df['id'] = df.index + 1
df['full_name'] = df['first_name'] + ' ' + df['last_name']

## 8. Save

Grouping labels are attached in `01-4_var_grouping.ipynb`.


In [ ]:
df.to_csv("./processed_job_data_0102_with_exp_pred_with_names.csv", index=False)
print(len(df), "rows written")
